# SUS analysis

Use this notebook to score complete System Usability Scale (SUS) responses and summarize results by round and build. Administer the authorized, validated 10-item SUS wording unchanged; this notebook does not reproduce it.

Keep the response CSV in approved restricted storage outside this repository. Do not commit participant-level data, identifiers, or raw comments. Outputs are aggregate only.

## Input format

Use one row per anonymous session with `session_code`, `round`, `build`, and `item_1` through `item_10`. Each response must be an integer from 1 to 5. Blank or invalid answers make that response set incomplete; no response is imputed. Set `DATA_PATH` to the approved CSV outside the repository.

In [ ]:
import csv
import statistics
from pathlib import Path

DATA_PATH = None  # Example: Path(r"C:\restricted\sus_responses.csv")
REQUIRED_COLUMNS = ["session_code", "round", "build"] + [
    f"item_{number}" for number in range(1, 11)
]

In [ ]:
def score_responses(responses):
    """Score ten valid answers, or return None without imputing missing data."""
    if len(responses) != 10 or any(
        not isinstance(value, int) or not 1 <= value <= 5
        for value in responses
    ):
        return None
    adjusted = [
        value - 1 if item_number % 2 else 5 - value
        for item_number, value in enumerate(responses, start=1)
    ]
    return sum(adjusted) * 2.5


def summarize_csv(path):
    groups = {}
    with Path(path).open(newline="", encoding="utf-8-sig") as source:
        reader = csv.DictReader(source)
        missing = set(REQUIRED_COLUMNS) - set(reader.fieldnames or [])
        if missing:
            raise ValueError("Missing required columns: " + ", ".join(sorted(missing)))
        for row in reader:
            round_name = (row["round"] or "").strip()
            build = (row["build"] or "").strip()
            session_code = (row["session_code"] or "").strip()
            if not round_name or not build or not session_code:
                raise ValueError("Every row needs an anonymous session code, round, and build.")
            responses = []
            for number in range(1, 11):
                raw = (row[f"item_{number}"] or "").strip()
                try:
                    value = int(raw) if raw else None
                except ValueError:
                    value = None
                responses.append(value if value is not None and 1 <= value <= 5 else None)
            key = (round_name, build)
            group = groups.setdefault(key, {"attempted": 0, "scores": []})
            group["attempted"] += 1
            score = score_responses(responses)
            if score is not None:
                group["scores"].append(score)

    summaries = []
    for (round_name, build), group in sorted(groups.items()):
        scores = group["scores"]
        summaries.append({
            "round": round_name,
            "build": build,
            "attempted": group["attempted"],
            "complete": len(scores),
            "incomplete": group["attempted"] - len(scores),
            "mean": statistics.mean(scores) if scores else None,
            "median": statistics.median(scores) if scores else None,
            "range": (min(scores), max(scores)) if scores else None,
        })
    return summaries


if DATA_PATH is None:
    print("Set DATA_PATH to the approved CSV outside this repository, then run this cell.")
else:
    print("Aggregate SUS results by round and build:")
    for result in summarize_csv(DATA_PATH):
        print(result)

## Interpretation and limitations

- A SUS score is on a 0–100 scale; it is **not a percentage**.
- Report completed-score counts alongside mean, median, and range. Incomplete records are not scored or imputed.
- Interpret SUS alongside task success, comprehension, and anonymized observations. SUS alone does not establish comprehension, accessibility, or fairness.
- Small usability rounds are descriptive. Do not claim statistical significance or publish individual scores or identifying subgroup summaries.